Symmetry of chemistry objects
=========================

This notebook introduces the usage of posym to determine the symmetry of molecular orbitals and wave functions 

Setup environment
----------------------

Install required packages

In [ ]:
try:
    import posym
except ImportError:
    ! pip install git+https://github.com/abelcarreras/posym.git
try:
    import pyscf
except ImportError:
    ! pip install pyscf
try:
    import matplotlib
except ImportError:
    ! pip install matplotlib

Import required modules

In [ ]:
from posym import SymmetryMolecule, SymmetryGaussianLinear, SymmetrySingleDeterminant
from posym.tools import get_basis_set_pyscf, build_density, build_orbital
from pyscf import gto, scf
import numpy as np
import matplotlib.pyplot as plt

Compute the electronic structure
---------------------------------------

Compute the electronic structure of the water molecule using pySCF

In [ ]:
r = 1  # O-H distance
alpha = np.deg2rad(104.5)  # H-O-H angle

mol_pyscf = gto.M(atom=[['O', [0, 0, 0]],
                        ['H', [-r, 0, 0]],
                        ['H', [r*np.cos(np.pi - alpha), r*np.sin(np.pi - alpha), 0]]],
                  basis='3-21g',
                  charge=0,
                  spin=0)

In [ ]:
pyscf_scf = scf.RHF(mol_pyscf).run()

Extract the molecular orbitals

In [ ]:
mo_coefficients = pyscf_scf.mo_coeff.T
overlap_matrix = pyscf_scf.get_ovlp(mol_pyscf)

Properties of the molecule

In [ ]:
print('n_atoms: ', mol_pyscf.natm)
print('n_orbitals: ', mol_pyscf.nao)
print('n_electrons: ', mol_pyscf.nelectron)

Geometry symmetry
-----------------

Analyze the symmetry of the molecular structure

In [ ]:
geom_sym = SymmetryMolecule('c2v', mol_pyscf.atom_coords(), [mol_pyscf.atom_symbol(i) for i in range(mol_pyscf.natm)])
print('geometry CSM: {:6.3f}'.format(geom_sym.measure))

Molecular orbitals symmetry
---------------------------

Obtain the basis functions to construct the molecular orbitals

In [ ]:
basis_set = get_basis_set_pyscf(mol_pyscf)

Construct the molecular orbitals from the basis functions and the MO coefficients

In [ ]:
orbitals = []
for i, orbital_vect in enumerate(mo_coefficients):
    orb = build_orbital(basis_set, orbital_vect)
    orbitals.append(orb)

Plot the molecular orbitals

In [ ]:
xrange = np.linspace(-4, 4, 100)
yrange = np.zeros_like(xrange)
zrange = np.zeros_like(xrange)

orbital_2 = orbitals[1]
orbital_3 = orbitals[2]
orbital_4 = orbitals[3]

plt.plot(xrange, orbital_2(xrange, yrange, zrange), label='orbital 2')
plt.plot(xrange, orbital_3(xrange, yrange, zrange), label='orbital 3')
plt.plot(xrange, orbital_4(xrange, yrange, zrange), label='orbital 4')
plt.xlabel('Borh')
plt.legend()

In [ ]:
x = np.linspace(-3, 2, 50)
y = np.linspace(-2, 3, 50)

X, Y = np.meshgrid(x, y)

for i, orbital in enumerate(orbitals):
    if i < 5 or i > 9:
        continue
    Z = orbital(X, Y, np.zeros_like(X))
    plt.imshow(Z, interpolation='bilinear', origin='lower', cmap='seismic')
    plt.title('orbital {}'.format(i+1))
    plt.xticks([])
    plt.yticks([])
    plt.figure()

Determine the symmetry of the molecular orbitals

In [ ]:
sym_orbitals = []
for i, orbital in enumerate(orbitals):
    sym_orb = SymmetryGaussianLinear('c2v', orbital,
                                     orientation_angles=geom_sym.orientation_angles,
                                     center=geom_sym.center
                                     )
    sym_orbitals.append(sym_orb)
    print('orbital {}: {}'.format(i+1, sym_orb))

Wave function symmetry
----------------------

In [ ]:
wf_alpha = sym_orbitals[7] * sym_orbitals[8] * sym_orbitals[9]
wf_beta = sym_orbitals[7] * sym_orbitals[8] * sym_orbitals[9]
wf_sym = wf_alpha * wf_beta

print('Configuration 1: ', wf_sym)

In [ ]:
wf_alpha = sym_orbitals[7] * sym_orbitals[8] * sym_orbitals[9]
wf_beta = sym_orbitals[7] * sym_orbitals[8] * sym_orbitals[10]
wf_sym = wf_alpha * wf_beta

print('Configuration 2: ', wf_sym)

In [ ]:
wf_sym = SymmetrySingleDeterminant('c2v',
                                   alpha_orbitals=[orbitals[7], orbitals[8], orbitals[9]],
                                   beta_orbitals=[orbitals[7], orbitals[8], orbitals[9]])
print('Configuration 1: ', wf_sym)

In [ ]:
wf_sym = SymmetrySingleDeterminant('c2v',
                                   alpha_orbitals=[orbitals[7], orbitals[8], orbitals[9]],
                                   beta_orbitals=[orbitals[7], orbitals[8], orbitals[10]])
print('Configuration 2: ', wf_sym)

Electronic density symmetry
----------------------------

In [ ]:
density_matrix_mo = np.diag([2.0]*(mol_pyscf.nelectron//2) + [0.0]*(mol_pyscf.nao-mol_pyscf.nelectron//2))
print(density_matrix_mo)
density_matrix_ao = mo_coefficients.T @ density_matrix_mo @ mo_coefficients

In [ ]:
f_density = build_density(basis_set, density_matrix_ao)

print('\ndensity integral: {:5.2f}'.format(f_density.integrate))

sm_dens = SymmetryGaussianLinear('c2v', f_density,
                                 orientation_angles=geom_sym.orientation_angles,
                                 center=geom_sym.center
                                 )
print(sm_dens)